In [1]:
from dotenv import load_dotenv
from typing import Optional
import httpx
import os

from mcp.server.mcpserver import MCPServer

mcp = MCPServer(
    name="OpenProject Management", 
    log_level="DEBUG"
)

OPENPROJECT_APIROOT = os.getenv("OPENPROJECT_APIROOT")
OPENPROJECT_TOKEN = os.getenv("OPENPROJECT_TOKEN")

class OpenProjectClient:
    def __init__(self):
        self._client: Optional[httpx.AsyncClient] = None

    def get_client(self) -> httpx.AsyncClient:
        if self._client is None or self._client.is_closed:
            self._client = httpx.AsyncClient(
                base_url=OPENPROJECT_APIROOT,
                auth=httpx.BasicAuth("apikey", OPENPROJECT_TOKEN),
                headers={
                    "Content-Type": "application/json",
                    "Accept": "application/json",
                },
                timeout=10.0,
            )
        return self._client

    async def close(self):
        await self._client.aclose()


op_server = OpenProjectClient()

In [9]:
from typing import List, Dict, Optional

async def list_project_workpackages(project_id: int) -> List[Dict]:
    http = op_server.get_client()
    response = await http.get(f"/api/v3/projects/{project_id}/work_packages", params=
                              {"pageSize": 1000, "offset": 1})
    
    response.raise_for_status()
    data = response.json()

    total = data.get("total", 0)
    elements = data["_embedded"]["elements"]
    
    all_elements = elements
    page_size = 1000
    offset = 2
    while len(all_elements) < total:
        resp = await http.get(
            f"/api/v3/projects/{project_id}/work_packages",
            params={"pageSize": page_size, "offset": offset}
        )
        resp.raise_for_status()
        page_data = resp.json()
        all_elements.extend(page_data["_embedded"]["elements"])
        offset += 1

    result = []
    for item in data["_embedded"]["elements"]:
        work_package = {}
        work_package["id"] = item.get("id", "")
        work_package["subject"] = item.get("subject", "")
        work_package["description"] = item.get("description", "")
        work_package["startDate"] = item.get("startDate", "")
        work_package["dueDate"] = item.get("dueDate", "")
        work_package["percentageDone"] = item.get("percentageDone", "")
        work_package["createdAt"] = item.get("createdAt", "")
        work_package["updatedAt"] = item.get("updatedAt", "")
        work_package["author"] = item.get("_links", {}).get("author", {}).get("title", {})
        work_package["assignee"] = item.get("_links", {}).get("assignee", {}).get("title", {})
        work_package["status"] = item.get("_links", {}).get("status", {}).get("title", {})

        result.append(work_package)
    
    result.sort(key=lambda wp: wp.get("createdAt") or "", reverse=True)

    
    return result

In [13]:
import nest_asyncio
nest_asyncio.apply()

import logging

# Silence third-party HTTP client trace/debug logs
logging.getLogger("httpx").setLevel(logging.WARNING)
logging.getLogger("httpcore").setLevel(logging.WARNING)

response = await list_project_workpackages(project_id=3)

In [14]:
response

[{'id': 1122,
  'subject': 'Refined Fraud Engine model pipeline (fraud_engine_v2/src/fraud_engine.py) in TrueClaim',
  'description': {'format': 'markdown',
   'raw': '**Project:** Arca True Claim\n**Date:** 11-Sep-2026\n**Status:** Completed\n\n### Task Details:\n• Refined Fraud Engine model pipeline (fraud_engine_v2/src/fraud_engine.py) in TrueClaim',
   'html': '<p class="op-uc-p"><strong>Project:</strong> Arca True Claim<br>\n<strong>Date:</strong> 11-Sep-2026<br>\n<strong>Status:</strong> Completed</p>\n<h3 id="task-details" class="op-uc-h3">\n<a href="#task-details" aria-hidden="true" rel="noopener noreferrer" class="op-uc-link_permalink icon-link op-uc-link"></a>Task Details:</h3>\n<p class="op-uc-p">• Refined Fraud Engine model pipeline (fraud_engine_v2/src/fraud_engine.py) in TrueClaim</p>'},
  'startDate': None,
  'dueDate': None,
  'percentageDone': None,
  'createdAt': '2026-09-12T02:45:50.260Z',
  'updatedAt': '2026-09-12T02:45:50.287Z',
  'author': 'Vidit Khairkar',
  'as